In [3]:
# ML Training Pipeline: Build and Save Risk Model (.pkl)
# - Detects target automatically: classification on 'decision'/'is_suspicious' or regression on 'risk_score'
# - Preprocesses features and trains tuned models
# - Evaluates and saves a single Pipeline to models/risk_model.pkl

import os
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    mean_squared_error,
    r2_score,
)
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import joblib

# Optional: XGBoost for stronger accuracy if available
try:
    from xgboost import XGBClassifier, XGBRegressor
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

DATA_PATH = Path("./user_behavior_dataset_final.csv")
MODEL_DIR = Path("./models")
MODEL_PATH = MODEL_DIR / "risk_model.pkl"

assert DATA_PATH.exists(), f"Dataset not found at {DATA_PATH}"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Load dataset
raw = pd.read_csv(DATA_PATH)
print(f"Loaded dataset with shape: {raw.shape}")
print("Columns:", list(raw.columns))

# Standardize boolean-like columns if present
bool_candidates = ["device_change", "location_change", "is_suspicious"]
for col in bool_candidates:
    if col in raw.columns:
        raw[col] = (
            raw[col]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace({"true": 1, "false": 0, "yes": 1, "no": 0, "allow": 0, "block": 1})
        )
        raw[col] = pd.to_numeric(raw[col], errors="coerce").fillna(0).astype(int)

# Detect target column
if "decision" in raw.columns:
    task_type = "classification"
    target_col = "decision"
elif "is_suspicious" in raw.columns:
    task_type = "classification"
    target_col = "is_suspicious"
elif "risk_score" in raw.columns:
    task_type = "regression"
    target_col = "risk_score"
else:
    raise ValueError("Dataset must include 'decision', 'is_suspicious', or 'risk_score' as target.")

# Restrict features to backend-available fields for runtime compatibility
FEATURE_WHITELIST = [
    "login_hour",
    "device_change",
    "location_change",
    "actions_per_session",
    "data_access_count",
    "failed_attempts",
]

available = set(raw.columns)
feature_cols = [c for c in FEATURE_WHITELIST if c in available]
missing = [c for c in FEATURE_WHITELIST if c not in available]
if missing:
    print("Warning: missing columns from whitelist:", missing)

X = raw[feature_cols].copy()
y = raw[target_col].copy()

# Identify categorical vs numeric
categorical_cols = []
numeric_cols = []
for c in feature_cols:
    if X[c].dtype == "object":
        categorical_cols.append(c)
    else:
        numeric_cols.append(c)

print("Task:", task_type)
print("Features (numeric):", numeric_cols)
print("Features (categorical):", categorical_cols)

# Preprocessors
numeric_transformer = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

# Split
if task_type == "classification":
    strat = y if y.nunique() > 1 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=strat
    )
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

# Candidate models with light tuning
best_model = None
best_score = -np.inf
best_name = None

if task_type == "classification":
    candidates = []
    candidates.append((
        "RandomForestClassifier",
        RandomForestClassifier(random_state=42),
        {"clf__n_estimators": [200], "clf__max_depth": [None, 8, 16], "clf__min_samples_split": [2, 5]},
    ))
    candidates.append((
        "LogisticRegression",
        LogisticRegression(max_iter=300),
        {"clf__C": [0.5, 1.0, 2.0]},
    ))
    if XGB_AVAILABLE:
        candidates.append((
            "XGBClassifier",
            XGBClassifier(
                n_estimators=300,
                learning_rate=0.1,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                eval_metric="mlogloss",
            ),
            {"clf__max_depth": [4, 6, 8], "clf__learning_rate": [0.05, 0.1]},
        ))

    for name, base_clf, grid in candidates:
        pipe = Pipeline(steps=[("preprocess", preprocess), ("clf", base_clf)])
        search = GridSearchCV(pipe, grid, cv=3, n_jobs=-1, verbose=1)
        search.fit(X_train, y_train)
        preds = search.predict(X_test)
        score = f1_score(y_test, preds, average="macro")
        print(f"Candidate {name} F1(macro): {score:.4f}")
        if score > best_score:
            best_score = score
            best_model = search.best_estimator_
            best_name = name

    print(f"Selected model: {best_name} with F1(macro)={best_score:.4f}")
    print("Test Accuracy:", accuracy_score(y_test, best_model.predict(X_test)))
    print("Confusion Matrix:\n", confusion_matrix(y_test, best_model.predict(X_test)))
    print("\nClassification Report:\n", classification_report(y_test, best_model.predict(X_test)))

else:
    candidates = []
    candidates.append((
        "RandomForestRegressor",
        RandomForestRegressor(random_state=42),
        {"reg__n_estimators": [400], "reg__max_depth": [None, 8, 16], "reg__min_samples_split": [2, 5]},
    ))
    if XGB_AVAILABLE:
        candidates.append((
            "XGBRegressor",
            XGBRegressor(
                n_estimators=500,
                learning_rate=0.08,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
            ),
            {"reg__max_depth": [4, 6, 8], "reg__learning_rate": [0.05, 0.08]},
        ))

    for name, base_reg, grid in candidates:
        pipe = Pipeline(steps=[("preprocess", preprocess), ("reg", base_reg)])
        search = GridSearchCV(pipe, grid, cv=3, n_jobs=-1, verbose=1)
        search.fit(X_train, y_train)
        preds = search.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2 = r2_score(y_test, preds)
        print(f"Candidate {name} RMSE: {rmse:.3f} | R2: {r2:.3f}")
        score = r2
        if score > best_score:
            best_score = score
            best_model = search.best_estimator_
            best_name = name

    print(f"Selected model: {best_name} with R2={best_score:.4f}")

# Save model
joblib.dump(best_model, MODEL_PATH)
print(f"Saved model to: {MODEL_PATH}")

# Quick reload test
model_loaded = joblib.load(MODEL_PATH)
print("Reloaded model; predicting on 3 test rows...")
print(model_loaded.predict(X_test.iloc[:3]))

Loaded dataset with shape: (3000, 8)
Columns: ['login_hour', 'device_change', 'location_change', 'actions_per_session', 'data_access_count', 'session_duration', 'failed_attempts', 'is_suspicious']
Task: classification
Features (numeric): ['login_hour', 'device_change', 'location_change', 'actions_per_session', 'data_access_count', 'failed_attempts']
Features (categorical): []
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Candidate RandomForestClassifier F1(macro): 0.8382
Fitting 3 folds for each of 3 candidates, totalling 9 fits
Candidate LogisticRegression F1(macro): 0.7121
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Candidate XGBClassifier F1(macro): 0.8094
Selected model: RandomForestClassifier with F1(macro)=0.8382
Test Accuracy: 0.9083333333333333
Confusion Matrix:
 [[ 75   9]
 [ 46 470]]

Classification Report:
               precision    recall  f1-score   support

           0       0.62      0.89      0.73        84
           1       0.98      0.